In [ ]:
import mlflow
from mlflow import MlflowClient
import pandas as pd
import json
from datetime import datetime

# Подключение к MLflow
MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"✅ Подключено к MLflow: {MLFLOW_TRACKING_URI}")

In [ ]:
# Создаем или получаем эксперимент для промптов
experiment_name = "diabetes_prediction_prompts"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
    experiment = mlflow.get_experiment(experiment_id)
    print(f"✅ Создан новый эксперимент: {experiment_name}")
else:
    print(f"✅ Используем существующий эксперимент: {experiment_name}")

print(f"📊 Experiment ID: {experiment.experiment_id}")

In [ ]:
# Функция для создания промпта
def create_prompt(prompt_name, version, text, tags=None):
    with mlflow.start_run(run_name=f"{prompt_name}_v{version}"):
        # Логируем теги
        mlflow.set_tag("prompt_name", prompt_name)
        mlflow.set_tag("prompt_version", version)
        mlflow.set_tag("created_at", datetime.now().isoformat())
        
        if tags:
            for key, value in tags.items():
                mlflow.set_tag(key, value)
        
        # Логируем параметры
        mlflow.log_param("prompt_name", prompt_name)
        mlflow.log_param("version", version)
        
        # Логируем текст промпта
        mlflow.log_text(text, f"prompt_v{version}.txt")
        
        # Сохраняем структурированную информацию
        prompt_info = {
            "name": prompt_name,
            "version": version,
            "text": text,
            "tags": tags,
            "created_at": datetime.now().isoformat()
        }
        
        import tempfile
        import json
        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            json.dump(prompt_info, f, indent=2)
            mlflow.log_artifact(f.name)
        
        print(f"✅ Создан промпт: {prompt_name} v{version}")
        return mlflow.active_run().info.run_id

In [ ]:
# Версия 1: Базовый промпт
create_prompt(
    "diabetes_prediction",
    "1.0",
    """
    Предсказание диабета на основе данных пациента.
    
    Входные признаки:
    - age: возраст
    - sex: пол
    - bmi: индекс массы тела
    - bp: артериальное давление
    
    Модель: Random Forest
    """,
    {"author": "ML Team", "status": "production"}
)

# Версия 1.1: Расширенный промпт
create_prompt(
    "diabetes_prediction",
    "1.1",
    """
    Предсказание прогрессии диабета.
    
    Полный список признаков:
    - age: возраст (годы)
    - sex: пол (1 - мужской, 2 - женский)
    - bmi: индекс массы тела (kg/m²)
    - bp: среднее артериальное давление (mm Hg)
    - s1: общий сывороточный холестерин (mg/dL)
    - s2: LDL холестерин (mg/dL)
    - s3: HDL холестерин (mg/dL)
    - s4: отношение общего холестерина к HDL
    - s5: логарифм уровня триглицеридов
    - s6: уровень глюкозы в крови (mg/dL)
    
    Важность признаков:
    1. bmi (35.8%)
    2. s5 (23.3%)
    3. bp (8.9%)
    
    Модель: Random Forest с 100 деревьями
    """,
    {"author": "ML Team", "status": "production", "feature_importance": "true"}
)

# Версия 2.0: Промпт для XGBoost
create_prompt(
    "diabetes_prediction",
    "2.0",
    """
    Предсказание диабета с использованием XGBoost.
    
    Параметры модели:
    - learning_rate: 0.1
    - max_depth: 6
    - n_estimators: 200
    
    Метрики:
    - R2 на тесте: 0.48
    - RMSE: 54.8
    
    Доверительные интервалы доступны
    """,
    {"author": "ML Team", "status": "staging", "algorithm": "xgboost"}
)

In [ ]:
# Получаем все промпты
from mlflow.tracking import MlflowClient

client = MlflowClient()
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

print(f"Найдено {len(runs)} версий промптов")
runs[['run_id', 'tags.prompt_name', 'tags.prompt_version', 'tags.status']].head(10)

In [ ]:
# Получаем конкретную версию промпта
def get_prompt_text(run_id):
    client = MlflowClient()
    artifacts = client.list_artifacts(run_id)
    
    for artifact in artifacts:
        if artifact.path.endswith('.txt'):
            local_path = mlflow.artifacts.download_artifacts(
                run_id=run_id,
                artifact_path=artifact.path
            )
            with open(local_path, 'r', encoding='utf-8') as f:
                return f.read()
    return None

# Берем последнюю версию
latest_run = runs.iloc[0]
prompt_text = get_prompt_text(latest_run.run_id)
print(f"Последняя версия (v{latest_run['tags.prompt_version']}):\n")
print(prompt_text)

In [ ]:
# Сравнение версий
comparison = runs[['tags.prompt_version', 'tags.author', 'tags.status', 
                   'start_time']].copy()
comparison.columns = ['Версия', 'Автор', 'Статус', 'Дата создания']
comparison
